# 03｜個股與市場資料驗收、可選 GitHub Artifact

**執行環境：Google Colab（Python 3）**。會讀取 `janusChatGPT` 研究資料及使用 GitHub `main` 的**固定 SHA**，結果只寫入研究 run／報告目錄。不會修改來源 canonical、Risk ACTIVE、GCP、Registry、PG 或 Core/Mart。

請由上往下執行每一個 cell；首次使用先完成 Google Drive OAuth。


In [ ]:
# 此設定每支 Notebook 可獨立執行；中途更換 Colab Runtime 也能接續同一研究 run。
import sys, os, json, hashlib, subprocess, traceback
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
assert ROOT.is_dir(), '沒有找到給ai專用/janusChatGPT；不能改用個人用資料夾'
DATASET = ROOT/'資料源'/'JanusDataset_v3_202610'
MARKET = ROOT/'資料源'/'大盤產業_免費來源_2025H1_試跑'
RUN_HOME = ROOT/'模型研究'/'_colab_pipeline_runs'
ACTIVE = RUN_HOME/'ACTIVE_RUN.json'
REPO = Path('/content/janus-omniforge')
REPO_URL = 'https://github.com/tommylin15/janus-omniforge.git'
RUN_HOME.mkdir(parents=True, exist_ok=True)

def sh(*args, check=True, capture=False):
    return subprocess.run([str(x) for x in args],check=check,text=True,
                          capture_output=capture)

def sha_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    return h.hexdigest()

def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str,allow_nan=False)+'\n',encoding='utf-8')
    tmp.replace(path)

def read_json(path):return json.loads(Path(path).read_text(encoding='utf-8'))

def packages():
    # Colab 預裝 pandas/numpy/scikit-learn；只有缺少依賴才安裝。不引入付費服務。
    from importlib.util import find_spec
    missing=[(module,pip) for module,pip in [('pyarrow','pyarrow'),('sklearn','scikit-learn'),('xgboost','xgboost')]
             if find_spec(module) is None]
    if missing:sh(sys.executable,'-m','pip','install','-q',*[p for _,p in missing])
    import pandas, numpy, pyarrow, sklearn, xgboost
    return {k:__import__(k).__version__ for k in ['pandas','numpy','pyarrow','sklearn','xgboost']}

def repo_git(*args):
    # 優先使用公開存取；若 repo 是 private，使用 Colab Secrets 的 GITHUB_TOKEN。
    cmd=['git',*[str(x) for x in args]]
    env=dict(os.environ,GIT_TERMINAL_PROMPT='0')
    result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    if result.returncode!=0:
        from google.colab import userdata
        token=userdata.get('GITHUB_TOKEN')
        if not token:raise RuntimeError('GIT_ACCESS_FAILED：repo 可能是 private；請設 Colab Secret GITHUB_TOKEN，或檢查存取權')
        import tempfile
        with tempfile.TemporaryDirectory() as folder:
            ask=Path(folder)/'askpass.sh'
            ask.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token;; *) printf %s "$GITHUB_TOKEN";; esac\n')
            ask.chmod(0o700)
            env.update(GIT_ASKPASS=str(ask),GITHUB_TOKEN=token)
            result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    if result.returncode:raise RuntimeError('GIT_COMMAND_FAILED：'+result.stderr[-500:])
    return result

def checkout(code_sha=None):
    if not (REPO/'.git').is_dir():repo_git('clone',REPO_URL,REPO)
    else:repo_git('-C',REPO,'fetch','origin','main')
    if code_sha:
        # 不能用最新 main 偷換固定研究程式版本。
        existing=sh('git','-C',REPO,'cat-file','-e',code_sha+'^{commit}',check=False)
        if existing.returncode:repo_git('-C',REPO,'fetch','origin',code_sha)
        repo_git('-C',REPO,'checkout','--detach','--force',code_sha)
    else:repo_git('-C',REPO,'checkout','--detach','--force','origin/main')
    actual=sh('git','-C',REPO,'rev-parse','HEAD',capture=True).stdout.strip()
    if code_sha and actual!=code_sha:raise RuntimeError('CODE_SHA_CHANGED')
    assert (REPO/'research/chatgpt/label_four_models.py').is_file()
    assert (REPO/'research/chatgpt/janus_train.py').is_file()
    code=str(REPO/'research/chatgpt')
    if code not in sys.path:sys.path.insert(0,code)
    return actual

def open_run():
    if not ACTIVE.is_file():raise RuntimeError('先執行 03_收檔驗收與同步.ipynb，建立研究 run')
    cfg=read_json(ACTIVE)
    out=RUN_HOME/cfg['run_id']
    if not out.is_dir():raise RuntimeError('RUN_FOLDER_MISSING')
    if not (DATASET/'canonical/CURRENT.json').is_file():raise RuntimeError('CURRENT_MISSING')
    if sha_file(DATASET/'canonical/CURRENT.json')!=cfg['current_sha256']:
        raise RuntimeError('CURRENT_CHANGED：來源版本已切換，請勿混用研究產物')
    manifest=DATASET/cfg['manifest_rel']
    if sha_file(manifest)!=cfg['manifest_sha256']:
        raise RuntimeError('MANIFEST_CHANGED：固定 release 有變動')
    checkout(cfg['git_sha'])
    return cfg,out

def save_stage(out,n,status,details):
    obj={'stage':n,'status':status,'updated_at_utc':datetime.now(timezone.utc).isoformat(),
         'publication_allowed':False,'details':details}
    write_json(Path(out)/f'{n}_status.json',obj)
    print(json.dumps(obj,ensure_ascii=False,indent=2,default=str)[:12000]);return obj

def stage(out,n):
    p=Path(out)/f'{n}_status.json'
    return read_json(p) if p.is_file() else {'stage':n,'status':'NOT_RUN'}

def ensure_stage(out,n,allow=('PASS',)):
    status=stage(out,n)['status']
    if status not in allow:raise RuntimeError(f'PREVIOUS_GATE_{n}={status}; allowed={allow}')

def keep_current(cfg):
    if sha_file(DATASET/'canonical/CURRENT.json')!=cfg['current_sha256']:
        raise RuntimeError('CURRENT_CHANGED_DURING_STAGE')

def fail_stage(out,n,exc):
    save_stage(out,n,'BLOCKED',{'error_type':type(exc).__name__, 'message':str(exc)[:1200]})
    raise exc

versions=packages()
print('Dependency versions:',versions)


In [ ]:
# 03：確認目前 Drive release，驗 155 個 manifest 引用檔案的完整 SHA + Parquet 筆數。
# 這支 Notebook 不補收、不修改來源，不會重新觸發 GitHub Actions。
import csv
import pyarrow.parquet as pq

# 已啟動過的 run 預設繼續；需要新 run 才手動改為 True。
NEW_RUN = False
if ACTIVE.exists() and not NEW_RUN:
    cfg,out=open_run()
else:
    git_sha=checkout()
    pointer=DATASET/'canonical/CURRENT.json'
    assert pointer.is_file(), 'CURRENT.json 缺失'
    c=read_json(pointer)
    assert c.get('format')=='parquet' and c.get('status')=='RESEARCH_DATA_ONLY'
    manifest=(DATASET/c['manifest']).resolve()
    assert manifest.is_relative_to(DATASET.resolve())
    assert manifest.parent.name==c.get('release')
    m=read_json(manifest)
    assert m.get('publication_allowed') is False
    assert m.get('release_id')==c['release'].removeprefix('release-')
    from datetime import datetime,timezone
    run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+sha_file(pointer)[:8]
    out=RUN_HOME/run_id;out.mkdir(parents=True,exist_ok=False)
    cfg={'run_id':run_id,'git_sha':git_sha,'current_sha256':sha_file(pointer),
         'manifest_rel':c['manifest'],'manifest_sha256':sha_file(manifest),
         'release':c['release'],'classification':'RESEARCH_ONLY','publication_allowed':False}
    write_json(out/'run_config.json',cfg);write_json(ACTIVE,cfg)

try:
    manifest=DATASET/cfg['manifest_rel'];m=read_json(manifest)
    records=[];issues=[];tables={};schema_warnings=[]
    release=manifest.parent.resolve()
    for name,expected in sorted(m['files'].items()):
        f=(release/name).resolve()
        if not f.is_relative_to(release):
            issues.append({'path':name,'code':'UNSAFE_PATH'});continue
        if not f.is_file():
            issues.append({'path':name,'code':'MISSING_FILE'});continue
        sha=sha_file(f)
        rows=None
        if name.endswith('.parquet'):
            md=pq.ParquetFile(f).metadata
            rows=md.num_rows
            assert rows==sum(md.row_group(i).num_rows for i in range(md.num_row_groups))
            # metadata 解讀：不宣稱逐列統計已驗。
            schema=str(pq.ParquetFile(f).schema_arrow)
            if name.startswith('revenue/') and 'unit_basis: int32' in schema.lower():
                schema_warnings.append({'path':name,'code':'WARN_UNIT_BASIS_INT32'})
        elif name.endswith('.csv'):
            with f.open(encoding='utf-8-sig',newline='') as stream:
                rows=max(0,sum(1 for _ in csv.reader(stream))-1)
        ok=(sha==expected['sha256'] and rows==expected['rows'])
        if not ok:issues.append({'path':name,'code':'HASH_OR_ROWS_MISMATCH',
                                 'actual_rows':rows,'declared_rows':expected['rows']})
        records.append({'path':name,'sha256':sha,'rows':rows,'declared_rows':expected['rows'],
                        'sha_pass':sha==expected['sha256'],'pass':ok})
        key=name.split('/')[0];tables[key]=tables.get(key,0)+1
    keep_current(cfg)
    summary={'release':cfg['release'],'git_sha':cfg['git_sha'],'files_declared':len(m['files']),
             'files_verified':sum(x['pass'] for x in records),'files_seen':len(records),
             'file_counts':tables,'issues':issues,'schema_warnings':schema_warnings,
             'github_artifact':'NOT_CHECKED_BY_DEFAULT','publication_allowed':False}
    write_json(out/'03_inventory.json',{'summary':summary,'files':records})
    status='PASS' if not issues and len(records)==len(m['files']) else 'BLOCKED'
    save_stage(out,'03',status,summary)
except Exception as exc:fail_stage(out,'03',exc)


In [ ]:
# 選用：若第 2 階段真的由 GitHub Actions 產生 artifact，輸入該 workflow run ID。
# 不設定就只驗 Drive 現有收檔。下載到獨立的 _colab_pipeline_runs/<run>/github_artifacts，絕不覆蓋來源。
GITHUB_RUN_ID = ''  # 例如：39912345678；留空代表不檢查 Actions artifact
if GITHUB_RUN_ID:
    import requests
    from google.colab import userdata
    token=userdata.get('GITHUB_TOKEN')  # 請在 Colab Secrets 放入，勿貼在 Notebook 文字
    if not token:raise RuntimeError('GITHUB_TOKEN_REQUIRED_FOR_ARTIFACTS')
    headers={'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json'}
    api=f'https://api.github.com/repos/tommylin15/janus-omniforge/actions/runs/{GITHUB_RUN_ID}/artifacts'
    q=requests.get(api,headers=headers,timeout=60);q.raise_for_status()
    artifacts=q.json().get('artifacts',[])
    folder=out/'github_artifacts';folder.mkdir(parents=True,exist_ok=True)
    receipt=[]
    for artifact in artifacts:
        aid=artifact['id']; dest=folder/f'{aid}.zip'
        if not dest.exists():
            with requests.get(f'https://api.github.com/repos/tommylin15/janus-omniforge/actions/artifacts/{aid}/zip',
                              headers=headers,timeout=120,stream=True) as stream:
                stream.raise_for_status()
                with dest.open('wb') as f:
                    for chunk in stream.iter_content(1024*1024):
                        if chunk:f.write(chunk)
        receipt.append({'id':aid,'name':artifact['name'],'zip_sha256':sha_file(dest),
                        'bytes':dest.stat().st_size})
    write_json(out/'03_github_artifact_receipt.json',{'run_id':GITHUB_RUN_ID,'artifacts':receipt,
                  'note':'Artifact ZIP 尚未自動升格或覆寫 Drive immutable source'})
    print('GitHub artifacts 存入研究驗收資料夾：',len(receipt))
else:
    print('GitHub artifact：NOT_CHECKED（選用檢查；不代表第 3 階段失敗）。')

# 無論是否指定 GitHub Run ID，最後一格都明確顯示真正的第 03 階段驗收收據。
print('\\n========== 第 03 階段最終驗收判定 ==========')
if 'out' not in globals():
    print('NOT_RUN：尚未建立驗收 run；請先依序執行上方程式區塊。')
else:
    receipt = stage(out, '03')
    details = receipt.get('details', {})
    summary = {
        'stage': '03',
        'status': receipt.get('status', 'NOT_RUN'),
        'release': details.get('release'),
        'files_declared': details.get('files_declared'),
        'files_verified': details.get('files_verified'),
        'files_seen': details.get('files_seen'),
        'issues_count': len(details.get('issues', [])),
        'schema_warnings': details.get('schema_warnings', []),
        'github_artifact': 'CHECKED_SEPARATELY' if GITHUB_RUN_ID else 'NOT_CHECKED_OPTIONAL',
        'stage_receipt': str(Path(out) / '03_status.json'),
        'publication_allowed': False
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    if summary['status'] == 'PASS':
        print('第 03 階段已 PASS，可以接續第 04 階段（不表示第 04～11 階段已通過）。')
    else:
        print('第 03 階段尚未 PASS；請先查看 03_status.json 的 details/錯誤。')

